In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats # For ANOVA/Kruskal-Wallis testing

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6) # Default plot size

# Load cleaned datasets
df_benin = pd.read_csv('C:/Users/Home-User/solar-challenge-week0/data/benin-malanville.csv', index_col='Timestamp', parse_dates=True)
df_sierra = pd.read_csv('C:/Users/Home-User/solar-challenge-week0/data/sierraleone-bumbuna.csv', index_col='Timestamp', parse_dates=True)
df_togo = pd.read_csv('C:/Users/Home-User/solar-challenge-week0/data/togo-dapaong_qc.csv', index_col='Timestamp', parse_dates=True)

# Create a 'Country' column for comparison
df_benin['Country'] = 'Benin'
df_sierra['Country'] = 'Sierra Leone'
df_togo['Country'] = 'Togo'

# Combine all datasets into one large DataFrame
df_all = pd.concat([df_benin, df_sierra, df_togo])

print(f"Combined Dataset Shape: {df_all.shape}")
print("Data loaded successfully.")

Combined Dataset Shape: (1576800, 20)
Data loaded successfully.


In [ ]:
core_metrics = ['GHI', 'DNI', 'DHI']

summary_table = df_combined_clean.groupby('Country')[core_metrics].agg(
    ['mean', 'median', 'std']
).round(2)

print("## Summary Statistics by Country (GHI, DNI, DHI)")
print(summary_table)

In [ ]:
# Melt the DataFrame to long format for easier plotting with Seaborn
df_melted = df_combined_clean.melt(id_vars=['Country'], value_vars=core_metrics,
                        var_name='Metric', value_name='Irradiance')

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)
fig.suptitle('Distribution of Cleaned Solar Irradiance Metrics by Country', fontsize=16)

for i, metric in enumerate(core_metrics):
    sns.boxplot(ax=axes[i], x='Country', y='Irradiance',
                data=df_melted[df_melted['Metric'] == metric],
                palette='coolwarm')
    axes[i].set_title(f'{metric} Distribution')
    axes[i].set_xlabel('')
    axes[i].set_ylabel(f'{metric} (W/m²)' if i == 0 else '')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# Extract GHI values for each country
benin_ghi = df_combined_clean[df_combined_clean['Country'] == 'Benin']['GHI'].dropna()
sierra_ghi = df_combined_clean[df_combined_clean['Country'] == 'Sierra Leone']['GHI'].dropna()
togo_ghi = df_combined_clean[df_combined_clean['Country'] == 'Togo']['GHI'].dropna()

# Kruskal-Wallis H-test
H_statistic, p_value = stats.kruskal(benin_ghi, sierra_ghi, togo_ghi)

print("\n## Kruskal-Wallis H-test Results (on GHI)")
print(f"H-Statistic: {H_statistic:.4f}")
print(f"P-value: {p_value:.10f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nConclusion: Reject the null hypothesis. The difference in median GHI is statistically significant (p < {alpha}).")
else:
    print(f"\nConclusion: Fail to reject the null hypothesis. There is no statistically significant difference in median GHI (p >= {alpha}).")

In [ ]:
# Calculate the mean GHI for each country
avg_ghi_rank = df_combined_clean.groupby('Country')['GHI'].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 5))
avg_ghi_rank.plot(kind='bar', color=['darkorange', 'skyblue', 'mediumseagreen'])
plt.title('Country Ranking by Average GHI (Solar Potential)')
plt.ylabel('Average GHI (W/m²)')
plt.xlabel('Country')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.5)
plt.show()

Key Cross-Country Observations
Highest Solar Potential (GHI): [Insert Country Name] has the highest overall solar potential, evidenced by the highest mean GHI of [Insert Mean GHI from table] W/m². This suggests it is the most resource-rich location.

Greatest Variability (Standard Deviation): The metric with the largest standard deviation relative to its mean is [Insert Metric, e.g., DNI] in [Insert Country Name]. This indicates the solar resource is the most inconsistent or variable in that country (likely due to inconsistent cloud cover).

Diffuse Component Comparison (DHI): [Insert Country Name] shows the highest median DHI. This suggests that a larger proportion of its solar energy is received indirectly (diffuse light), which can be caused by higher atmospheric haze, dust, or light cloud cover.